# 🇻🇳 ViMind: Quy Trình Huấn Luyện Toàn Diện Trên Kaggle GPU
Notebook này chứa trọn vẹn 2 giai đoạn huấn luyện:
1. **Giai đoạn 3:** Tiền huấn luyện (Pre-training) trên 405.000 bài viết tiếng Việt để học từ vựng và ngữ pháp.
2. **Giai đoạn 4:** Huấn luyện chỉ dẫn (SFT) trên 52.000 hội thoại để biến mô hình thành Trợ lý ảo đối thoại.

In [ ]:
# 1. Cài đặt các thư viện phụ thuộc
!pip install -r requirements.txt


In [ ]:
# 2. Kiểm tra phần cứng GPU T4 (16GB VRAM)
!nvidia-smi

In [ ]:
# 3. Chạy Dry-Run kiểm tra toàn bộ pipeline (Pre-train & SFT Masking)
!python trainer/test_dry_run.py
!python trainer/test_sft_dry_run.py

In [ ]:
# 4. [Giai đoạn 3] Bắt đầu Pre-training trên 405.000 tài liệu tiếng Việt
!python trainer/pretrain.py \
    --data_path dataset/pretrain_vi.jsonl \
    --tokenizer_dir model \
    --save_dir out \
    --save_weight vimind_26m \
    --batch_size 32 \
    --accumulation_steps 4 \
    --epochs 1 \
    --dtype float16 \
    --log_interval 50 \
    --save_interval 1000

In [ ]:
# 5. [Giai đoạn 4] Bắt đầu Supervised Fine-Tuning (SFT) từ trọng số Pre-train
!python trainer/train_sft.py \
    --data_path dataset/sft_vi.jsonl \
    --tokenizer_dir model \
    --from_pretrained out/vimind_26m_final \
    --save_dir out/sft \
    --save_weight vimind_sft \
    --batch_size 16 \
    --accumulation_steps 4 \
    --epochs 2 \
    --learning_rate 1e-4 \
    --dtype float16 \
    --log_interval 25 \
    --save_interval 500

In [ ]:
# 6. Kiểm tra câu trả lời của mô hình thành phẩm sau khi train SFT xong
!python -c "import torch; from transformers import AutoTokenizer; from model.model import ViMindForCausalLM; tok = AutoTokenizer.from_pretrained('out/sft/vimind_sft_final'); model = ViMindForCausalLM.from_pretrained('out/sft/vimind_sft_final').cuda(); prompt = tok.apply_chat_template([{'role': 'user', 'content': 'Xin chào, bạn là ai?'}], tokenize=False, add_generation_prompt=True); ids = tok(prompt, return_tensors='pt').input_ids.cuda(); out = model.generate(ids, max_new_tokens=100); print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))"